## **Tareas a resolver:**

**Clasificación binaria: mentira o verdad**

**Predicción del hablante: de qué país es el mensaje**

Antes de empezar a resolver las tareas vamos a intentar arreglar el problema del desbalance de clases

**1. Carga y análisis inicial del dataset**
- Cargar datos
- Expandir mensajes
- Analizar distribución de clases
- Identificar desbalance

In [13]:
import pandas as pd
import json

data = pd.read_parquet('data/train_preprocessed.parquet')
df_expanded = data.explode(["messages", "sender_labels", "receiver_labels"])
df_expanded.head()

,messages,sender_labels,receiver_labels,speakers,receivers,absolute_message_index,relative_message_index,seasons,years,game_score,game_score_delta,players,game_id,text_clean,tokens,lemmas
0,Germany!\n\nJust the person I want to speak wi...,True,True,"['italy', 'germany', 'italy', 'germany', 'ital...","['germany', 'italy', 'germany', 'italy', 'germ...","[74, 76, 86, 87, 89, 92, 97, 117, 119, 121, 12...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['Spring', 'Spring', 'Spring', 'Spring', 'Spri...","['1901', '1901', '1901', '1901', '1901', '1901...","['3', '3', '3', '3', '3', '3', '3', '3', '3', ...","['0', '0', '0', '0', '0', '0', '0', '0', '0', ...","['italy', 'germany']",1,germany just the person i want to speak with i...,"['germany', 'person', 'want', 'speak', 'somewh...","['germany', 'person', 'want', 'speak', 'somewh..."
1,"You've whet my appetite, Italy. What's the sug...",True,True,"['italy', 'germany', 'italy', 'germany', 'ital...","['germany', 'italy', 'germany', 'italy', 'germ...","[74, 76, 86, 87, 89, 92, 97, 117, 119, 121, 12...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['Spring', 'Spring', 'Spring', 'Spring', 'Spri...","['1901', '1901', '1901', '1901', '1901', '1901...","['3', '3', '3', '3', '3', '3', '3', '3', '3', ...","['0', '0', '0', '0', '0', '0', '0', '0', '0', ...","['italy', 'germany']",1,youve whet my appetite italy whats the suggestion,"['ve', 'whet', 'appetite', 'italy', 's', 'sugg...","['ve', 'whet', 'appetite', 'italy', 's', 'sugg..."
2,It seems like there are a lot of ways that cou...,True,True,"['italy', 'germany', 'italy', 'germany', 'ital...","['germany', 'italy', 'germany', 'italy', 'germ...","[74, 76, 86, 87, 89, 92, 97, 117, 119, 121, 12...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['Spring', 'Spring', 'Spring', 'Spring', 'Spri...","['1901', '1901', '1901', '1901', '1901', '1901...","['3', '3', '3', '3', '3', '3', '3', '3', '3', ...","['0', '0', '0', '0', '0', '0', '0', '0', '0', ...","['italy', 'germany']",1,it seems like there are a lot of ways that cou...,"['like', 'lot', 'ways', 'wrong', 'nt', 'france...","['like', 'lot', 'way', 'wrong', 'not', 'france..."
3,"Yeah, I can’t say I’ve tried it and it works, ...",True,NOANNOTATION,"['italy', 'germany', 'italy', 'germany', 'ital...","['germany', 'italy', 'germany', 'italy', 'germ...","[74, 76, 86, 87, 89, 92, 97, 117, 119, 121, 12...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['Spring', 'Spring', 'Spring', 'Spring', 'Spri...","['1901', '1901', '1901', '1901', '1901', '1901...","['3', '3', '3', '3', '3', '3', '3', '3', '3', ...","['0', '0', '0', '0', '0', '0', '0', '0', '0', ...","['italy', 'germany']",1,yeah i can t say i ve tried it and it works ca...,"['yeah', 't', 've', 'tried', 'works', 'cause',...","['yeah', 't', 've', 'try', 'work', 'cause', 'v..."
4,I am just sensing that you don’t like this ide...,True,NOANNOTATION,"['italy', 'germany', 'italy', 'germany', 'ital...","['germany', 'italy', 'germany', 'italy', 'germ...","[74, 76, 86, 87, 89, 92, 97, 117, 119, 121, 12...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['Spring', 'Spring', 'Spring', 'Spring', 'Spri...","['1901', '1901', '1901', '1901', '1901', '1901...","['3', '3', '3', '3', '3', '3', '3', '3', '3', ...","['0', '0', '0', '0', '0', '0', '0', '0', '0', ...","['italy', 'germany']",1,i am just sensing that you don t like this ide...,"['sensing', 'don', 't', 'like', 'idea', 'shall...","['sense', 'don', 't', 'like', 'idea', 'shall',..."


**2. Análisis del desbalance de clases**
- Distribución de `sender_labels`
- Distribución de `receiver_labels`

In [10]:
print("Distribución sender_labels:")
print(df_expanded["sender_labels"].value_counts())
print("\nDistribución receiver_labels:")
print(df_expanded["receiver_labels"].value_counts())

Distribución sender_labels:
sender_labels
True     11372
False      522
Name: count, dtype: int64

Distribución receiver_labels:
receiver_labels
True            10390
NOANNOTATION      989
False             515
Name: count, dtype: int64


**3. Corrección del desbalance de clases**
- Uso de *class weights*
- Preparación para entrenamiento

In [14]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# -------- sender_labels (binario) --------
y_sender = df_expanded["sender_labels"]

sender_class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_sender),
    y=y_sender
)

print("\nClass weights para sender_labels:")
print(sender_class_weights)


# -------- receiver_labels (multiclase) --------
y_receiver = df_expanded["receiver_labels"]

receiver_class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_receiver),
    y=y_receiver
)

print("\nClass weights para receiver_labels:")
print(receiver_class_weights)


Class weights para sender_labels:
[11.39272031  0.52295111]

Class weights para receiver_labels:
[7.69838188 4.00876306 0.38158486]


**Resultados**

Durante el entrenamiento de los modelos, utilizaremos estos pesos para que la función de pérdida multiplique el error de cada ejemplo por el peso correspondiente a su clase. De esta manera, el optimizador ajustará los parámetros del modelo para minimizar la pérdida ponderada, lo que mejora significativamente la capacidad del modelo para predecir correctamente las clases minoritarias.

## **Shallow ML**
En esta sección vamos a intentar resolver las tareas con técnicas de shallow ML

### **1. Clasificación binaria**
   

In [6]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report
from scipy.sparse import load_npz
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from imblearn.over_sampling import RandomOverSampler
import joblib

# --------------------------
# 1. Cargar datos
# --------------------------
X_tfidf = load_npz("diplomacy/models/representations/X_tfidf_train.npz")
X_bow   = load_npz("diplomacy/models/representations/X_bow_train.npz")

df = pd.read_parquet("data/train_preprocessed.parquet")

# Convertir etiquetas sender_labels a binario 0/1
y = df["sender_labels"].astype(str).str.lower().map({"true": 1, "false": 0})

print("Distribución de clases:")
print(y.value_counts())

# --------------------------
# 2. Pesos de clase
# --------------------------
class_weight_sender = {
    0: 11.39272031,   # False = minoritaria
    1: 0.52295111     # True = mayoritaria
}

# Para XGBoost usamos scale_pos_weight como recomendación oficial
pos_weight = class_weight_sender[0] / class_weight_sender[1]
print("\nscale_pos_weight =", pos_weight)

# --------------------------
# 3. División train/val
# --------------------------
X_train_tfidf, X_val_tfidf, y_train, y_val = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42, stratify=y
)

X_train_bow, X_val_bow, _, _ = train_test_split(
    X_bow, y, test_size=0.2, random_state=42, stratify=y
)

# ==================================================
# 4. LOGISTIC REGRESSION (TF-IDF)
# ==================================================
lr = LogisticRegression(
    max_iter=3000,
    class_weight=class_weight_sender
)

lr.fit(X_train_tfidf, y_train)
y_pred_lr = lr.predict(X_val_tfidf)

print("\n=== Logistic Regression (TF-IDF) ===")
print(classification_report(y_val, y_pred_lr))

joblib.dump(lr, "diplomacy/models/shallow/logreg_tfidf_weighted.joblib")

# ==================================================
# 5. LINEAR SVM (BoW)
# ==================================================
svm = LinearSVC(
    class_weight=class_weight_sender,
    max_iter=3000
)

svm.fit(X_train_bow, y_train)
y_pred_svm = svm.predict(X_val_bow)

print("\n=== Linear SVM (BoW) ===")
print(classification_report(y_val, y_pred_svm))

joblib.dump(svm, "diplomacy/models/shallow/svm_bow_weighted.joblib")

# ==================================================
# 6. XGBOOST (TF-IDF) – con oversampling para evitar collapse
# ==================================================
print("\nAplicando oversampling SOLO para XGBoost...")

ros = RandomOverSampler(random_state=42)
X_train_tfidf_bal, y_train_bal = ros.fit_resample(X_train_tfidf, y_train)

print("Distribución balanceada para XGBoost:")
print(pd.Series(y_train_bal).value_counts())

xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    scale_pos_weight=pos_weight,
    n_estimators=400,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    n_jobs=-1
)

xgb.fit(X_train_tfidf_bal, y_train_bal)
y_pred_xgb = xgb.predict(X_val_tfidf)

print("\n=== XGBoost (TF-IDF + Oversampling + scale_pos_weight) ===")
print(classification_report(y_val, y_pred_xgb))

joblib.dump(xgb, "diplomacy/models/shallow/xgb_tfidf_weighted.joblib")

Distribución de clases:
sender_labels
1    11372
0      522
Name: count, dtype: int64

scale_pos_weight = 21.78544053573191

=== Logistic Regression (TF-IDF) ===
              precision    recall  f1-score   support

           0       0.11      0.22      0.15       104
           1       0.96      0.92      0.94      2275

    accuracy                           0.89      2379
   macro avg       0.54      0.57      0.55      2379
weighted avg       0.93      0.89      0.91      2379


=== Linear SVM (BoW) ===
              precision    recall  f1-score   support

           0       0.07      0.09      0.08       104
           1       0.96      0.95      0.95      2275

    accuracy                           0.91      2379
   macro avg       0.52      0.52      0.52      2379
weighted avg       0.92      0.91      0.92      2379


Aplicando oversampling SOLO para XGBoost...
Distribución balanceada para XGBoost:
sender_labels
1    9097
0    9097
Name: count, dtype: int64

=== XGBoost (T

['diplomacy/models/shallow/xgb_tfidf_weighted.joblib']

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split, GridSearchCV
from scipy.sparse import load_npz, hstack
from imblearn.over_sampling import SMOTE
from gensim.models import Word2Vec
from sklearn.preprocessing import StandardScaler
import joblib

# ======================================
# 1. CARGA DE DATOS Y ETIQUETAS
# ======================================
print("Cargando datos...")
X_tfidf = load_npz("diplomacy/models/representations/X_tfidf_train.npz")
df = pd.read_parquet("data/train_preprocessed.parquet")

# Etiquetas en formato binario
y = df["sender_labels"].astype(str).str.lower().map({"true": 1, "false": 0})

# Tus class weights calculados previamente
sender_class_weights = {
    0: 11.39272031,
    1: 0.52295111
}

# Cargar Word2Vec
w2v = Word2Vec.load("diplomacy/models/embeddings/word2vec.model")

# ======================================
# 2. GENERAR WORD2VEC PROMEDIADO POR TEXTO
# ======================================
def text_to_w2v(tokens):
    tokens = eval(tokens) if isinstance(tokens, str) else tokens
    vecs = [w2v.wv[word] for word in tokens if word in w2v.wv]
    if len(vecs) == 0:
        return np.zeros(w2v.vector_size)
    return np.mean(vecs, axis=0)

print("Generando embeddings Word2Vec...")
w2v_features = np.vstack(df["tokens"].apply(text_to_w2v).values)

# Escalado para combinar con TF-IDF
scaler = StandardScaler()
w2v_features_scaled = scaler.fit_transform(w2v_features)

# Convertir a sparse y concatenar con TF-IDF
w2v_sparse = np.nan_to_num(w2v_features_scaled)
X_combined = hstack([X_tfidf, w2v_sparse])

# ======================================
# 3. TRAIN/VAL SPLIT
# ======================================
X_train, X_val, y_train, y_val = train_test_split(
    X_combined, y, test_size=0.2, random_state=42, stratify=y
)

# ======================================
# 4. SMOTE (mejor que RandomOverSampler)
# ======================================
print("Aplicando SMOTE...")
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Distribución tras SMOTE:")
print(pd.Series(y_train_smote).value_counts())

# ======================================
# 5. GRIDSEARCH PARA LOGISTIC REGRESSION
# ======================================
print("\nBuscando mejores hiperparámetros (LogReg)...")

params = {
    "C": [0.1, 1, 5],
    "penalty": ["l2"],
    "solver": ["liblinear", "lbfgs"],
    "class_weight": [
        None,
        "balanced",
        sender_class_weights
    ]
}

grid = GridSearchCV(
    LogisticRegression(max_iter=3000),
    param_grid=params,
    scoring="f1_macro",
    cv=3,
    n_jobs=-1
)

grid.fit(X_train_smote, y_train_smote)

print("Mejores parámetros:", grid.best_params_)

best_lr = grid.best_estimator_
y_pred_lr = best_lr.predict(X_val)

print("\nResultados Logistic Regression + SMOTE + Word2Vec + GridSearch:")
print(classification_report(y_val, y_pred_lr))

joblib.dump(best_lr, "diplomacy/models/shallow/logreg_advanced.joblib")

# ======================================
# 6. BUSCAR UMBRAL ÓPTIMO PARA CLASE 0
# ======================================
print("\nBuscando umbral óptimo para clase minoritaria...")

y_probs = best_lr.predict_proba(X_val)[:, 1]

best_thr, best_f1 = 0, 0
for thr in np.arange(0.1, 0.9, 0.05):
    pred = (y_probs >= thr).astype(int)
    f1 = f1_score(y_val, pred, pos_label=0)
    if f1 > best_f1:
        best_f1, best_thr = f1, thr

print(f"Mejor umbral = {best_thr:.2f} con F1(minoría) = {best_f1:.3f}")

final_pred = (y_probs >= best_thr).astype(int)

print("\nResultados Logistic Regression con umbral optimizado:")
print(classification_report(y_val, final_pred))

Cargando datos...
Generando embeddings Word2Vec...
Aplicando SMOTE...
Distribución tras SMOTE:
sender_labels
1    9097
0    9097
Name: count, dtype: int64

Buscando mejores hiperparámetros (LogReg)...
Mejores parámetros: {'C': 5, 'class_weight': 'balanced', 'penalty': 'l2', 'solver': 'lbfgs'}

Resultados Logistic Regression + SMOTE + Word2Vec + GridSearch:
              precision    recall  f1-score   support

           0       0.14      0.16      0.15       104
           1       0.96      0.96      0.96      2275

    accuracy                           0.92      2379
   macro avg       0.55      0.56      0.56      2379
weighted avg       0.93      0.92      0.92      2379


Buscando umbral óptimo para clase minoritaria...
Mejor umbral = 0.55 con F1(minoría) = 0.158

Resultados Logistic Regression con umbral optimizado:
              precision    recall  f1-score   support

           0       0.14      0.18      0.16       104
           1       0.96      0.95      0.96      2275

 

### **2. Predicción del hablante**


## **CNNs o Redes Recurrentes**

En esta sección vamos a entrenar modelos CNN para resolver las tareas

### **1. Clasificación binaria**
   

### **2. Predicción del hablante**
